# Log-Likelihood Ratio (LLR) Test

Compares two nested models (e.g., cloudfree vs mixture) by computing:

$$T = \log_{10}\left(\frac{r_{\text{complex}}}{r_{\text{simple}}}\right)$$

where $r = \max_\theta \frac{q(\theta|x)}{\pi(\theta)}$ is the MAP of the posterior-to-prior ratio.

A null distribution is built by simulating under the simpler model.

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.utils.checkpoint import load_checkpoint, load_model_state
from sbi4atmret.evaluation.llr import LLREvaluator, LLRResult
from zuko.distributions import BoxUniform

## 1. Load Two Competing Models

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Simple model (denominator): e.g., cloudfree ---
config_den_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"
with open(config_den_path) as f:
    config_den = BaseConfig(**yaml.safe_load(f))

model_den = BaseModel(config_den).build()
cp_den = load_checkpoint(Path("path/to/cloudfree/checkpoints/latest.pt"), device)
load_model_state(model_den.estimator, cp_den)
model_den.estimator.to(device).eval()

# Prior for simple model
lower_den, upper_den = config_den.get_parameter_bounds()
prior_den = BoxUniform(torch.tensor(lower_den), torch.tensor(upper_den)).to(device)
bounds_den = list(zip(lower_den, upper_den))

print(f"Simple model: {config_den.get_no_of_params()} params")

In [ ]:
# --- Complex model (numerator): e.g., mixture/cloudy ---
# config_num_path = project_root / "experiments/config_MiriGeminiHST_mixture.yaml"
# with open(config_num_path) as f:
#     config_num = BaseConfig(**yaml.safe_load(f))

# model_num = BaseModel(config_num).build()
# cp_num = load_checkpoint(Path("path/to/mixture/checkpoints/latest.pt"), device)
# load_model_state(model_num.estimator, cp_num)
# model_num.estimator.to(device).eval()

# lower_num, upper_num = config_num.get_parameter_bounds()
# prior_num = BoxUniform(torch.tensor(lower_num), torch.tensor(upper_num)).to(device)
# bounds_num = list(zip(lower_num, upper_num))

# print(f"Complex model: {config_num.get_no_of_params()} params")

## 2. Load Observation

In [ ]:
# x_obs = torch.from_numpy(observation.full_observation).float()
# print(f"Observation shape: {x_obs.shape}")

## 3. Compute LLR on Observation

In [ ]:
llr_eval = LLREvaluator()

save_path = Path("llr_results")
save_path.mkdir(exist_ok=True)

# Compute T_obs only (no null distribution yet)
T_obs, theta_num, theta_den = llr_eval.compute_llr(
    x_obs=x_obs,
    net_num=model_num.estimator,
    net_den=model_den.estimator,
    prior_num=prior_num,
    prior_den=prior_den,
    bounds_num=bounds_num,
    bounds_den=bounds_den,
    device=device,
)

print(f"T_obs = {T_obs:.4f}")
print(f"  θ_MAP (complex): {theta_num[:5]}...")
print(f"  θ_MAP (simple):  {theta_den[:5]}...")

## 4. Build Null Distribution

Simulate observations under the simpler model (cloudfree) and compute T for each.
This gives the distribution of T under H0 (simple model is correct).

In [ ]:
# Define noise function matching your noise model
# def noise_fn(x, theta):
#     """Add heteroscedastic noise to simulated spectrum."""
#     b = theta[-1]  # b-factor (last param for MIRI)
#     sigma = observation.obs_noise["miri"]
#     sigma_new = torch.sqrt(torch.tensor(sigma)**2 + 10**b)
#     error = sigma_new * torch.randn_like(x) * domain.scale
#     return x + error

# # Get the simpler model's simulator
# simulator_den = domain.simulator_dict["cloudfree_miri"]  # or whichever

# result = llr_eval.run(
#     x_obs=x_obs,
#     net_num=model_num.estimator,
#     net_den=model_den.estimator,
#     prior_num=prior_num,
#     prior_den=prior_den,
#     bounds_num=bounds_num,
#     bounds_den=bounds_den,
#     simulator_den=simulator_den,
#     noise_fn=noise_fn,
#     n_null=200,
#     device=device,
#     save_path=save_path,
# )

# print(f"T_obs = {result.T_obs:.4f}")
# print(f"p-value = {result.p_value:.4f}")

## 5. Plot Results

In [ ]:
# If you ran the full test:
# result.figure  # already saved as llr_test.pdf
# plt.show()

# Or load saved null distribution:
# T_null = np.loadtxt(save_path / "T_null.csv")
# T_obs = np.loadtxt(save_path / "T_obs.txt")[0]

# fig, ax = plt.subplots(figsize=(7, 4))
# ax.hist(T_null, bins=30, alpha=0.6, color="steelblue", label="Null (simple model)")
# ax.axvline(T_obs, color="red", linewidth=2, linestyle="--",
#            label=f"T_obs = {T_obs:.3f}")
# p = (T_null >= T_obs).mean()
# ax.set_title(f"LLR Test: p = {p:.4f}")
# ax.set_xlabel(r"$T = \log_{10}(r_{complex} / r_{simple})$")
# ax.legend()
# plt.tight_layout()
# plt.savefig(save_path / "llr_test.pdf", bbox_inches="tight")
# plt.show()

## 6. Interpretation

- **T_obs > 0**: Complex model fits better
- **T_obs ≈ 0**: Models are indistinguishable
- **T_obs < 0**: Simple model fits better (Occam's razor)
- **p-value < 0.05**: Reject H0 (simple model), favor complex model

Note: Unlike a classifier-based Bayes factor, this approach directly
compares the posterior-to-prior ratios at their optima. It's more
expensive (requires optimization per observation) but doesn't need
training a separate classifier.